# Bidirectional Traversal Proof (2x2 Chain)

This notebook demonstrates:
1. Circuit -> Geometry propagation (pad widths)
2. Geometry -> Netlist propagation (routes populate net_info)
3. Geometry -> Circuit extraction (explicit extractor)
4. Round-trip (override circuit, rebuild, geometry updates)

In [2]:
from pathlib import Path
import sys

repo = Path.cwd()
src = repo / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qiskit_metal.toolbox_metal.design_dsl import build_design

yaml_path = repo / "2x2_4qubit.chain.metal.yaml"
design = build_design(yaml_path)
chain = design.metadata.get("dsl_chain", {})

# 1) Circuit -> geometry
pad_circuit = str(chain["circuit"]["Q1"]["pad_width"])
pad_geom = str(design.components["Q1"].options.get("pad_width"))
assert pad_geom == pad_circuit, "Circuit -> geometry propagation failed"

# 2) Geometry -> netlist
assert len(design.net_info) > 0, "Routes did not populate net_info"

# 3) Geometry -> circuit (extractor)
route_names = ["top_bus", "left_bus", "bottom_bus", "right_bus"]
route_lengths = {}
for name in route_names:
    if name in design.components:
        length = design.components[name].options.get("_actual_length")
        if length:
            route_lengths[name] = str(length)

chain.setdefault("circuit", {})["route_lengths"] = route_lengths
assert route_lengths, "No route lengths extracted from geometry"

# 4) Round-trip: override circuit and rebuild
override_circuit = dict(chain["circuit"])
for qubit in ("Q1", "Q2", "Q3", "Q4"):
    override_circuit[qubit] = dict(override_circuit[qubit])
    override_circuit[qubit]["pad_width"] = "450um"

design2 = build_design(yaml_path, overrides={"circuit": override_circuit})
pad_geom_2 = str(design2.components["Q1"].options.get("pad_width"))
assert pad_geom_2 == "450um", "Round-trip override failed"

print("PASS: circuit -> geometry")
print("PASS: geometry -> netlist")
print("PASS: geometry -> circuit (extractor)")
print("PASS: circuit override -> geometry (round-trip)")

11:54AM 50s WARNING [check_lengths]: For path table, component=top_bus, key=trace has short segments that could cause issues with fillet. Values in (17-18)  are index(es) in shapely geometry.
11:54AM 50s WARNING [check_lengths]: For path table, component=top_bus, key=cut has short segments that could cause issues with fillet. Values in (17-18)  are index(es) in shapely geometry.
11:54AM 50s WARNING [check_lengths]: For path table, component=top_bus, key=trace has short segments that could cause issues with fillet. Values in (17-18)  are index(es) in shapely geometry.
11:54AM 50s WARNING [check_lengths]: For path table, component=top_bus, key=cut has short segments that could cause issues with fillet. Values in (17-18)  are index(es) in shapely geometry.


PASS: circuit -> geometry
PASS: geometry -> netlist
PASS: geometry -> circuit (extractor)
PASS: circuit override -> geometry (round-trip)
